In [6]:
# == 0. Kaggle working-directory cleanup (run first, once per session) ========
# Clears stale output files from earlier attempts (e.g. the runs that produced
# 0-verified-claims before the claim_id normalization fix) so you always see
# FRESH results, not left-over files from a prior buggy run.
# Only touches files THIS notebook generates -- never touches attached input
# datasets under /kaggle/input (those are read-only anyway), and never deletes
# done_annotator_*.csv anywhere -- those are your real annotations, not generated
# outputs, even if an earlier run happened to leave them under ./annotation.
from pathlib import Path

_work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
_stale_patterns = [
    "iaa_summary.json",
    "scireqa_human_verified_subset.jsonl",
    "*_humanmerged.jsonl",
    "minicheck_calibration.json",
    "threeway_agreement.json",
    "table5_human_validation.csv",
]
removed = []
for pat in _stale_patterns:
    for f in _work.glob(pat):
        f.unlink()
        removed.append(str(f))

# Also clean the default local ANNOT_DIR ("annotation"), which cell 1 creates
# unconditionally via ANNOT_DIR.mkdir(exist_ok=True) even when you've pointed
# ANNOT_DIR at a real attached dataset elsewhere -- so it can fill up with
# stale sample_ids.json / claims_master.csv / to_annotate_*.csv from earlier
# attempts. done_annotator_*.csv is explicitly SKIPPED and just reported.
_default_annot = _work / "annotation"
_stale_input_names = ["sample_ids.json", "claims_master.csv"]
if _default_annot.exists():
    for name in _stale_input_names:
        f = _default_annot / name
        if f.exists():
            f.unlink()
            removed.append(str(f))
    for f in sorted(_default_annot.glob("to_annotate_*.csv")):
        f.unlink()
        removed.append(str(f))
    preserved = sorted(_default_annot.glob("done_*.csv"))
    if preserved:
        print(f"NOTE: found {len(preserved)} done_annotator_*.csv under {_default_annot} -- "
              f"left untouched (these look like real annotations, not generated output):")
        for p in preserved:
            print("  ", p)

print(f"Cleaned {_work.resolve()}" + (f" and {_default_annot}" if _default_annot.exists() else ""))
if removed:
    print(f"Removed {len(removed)} stale file(s):")
    for r in removed:
        print(" ", r)
else:
    print("Nothing to remove -- already clean.")


Cleaned /kaggle/working and /kaggle/working/annotation
Removed 5 stale file(s):
  /kaggle/working/annotation/sample_ids.json
  /kaggle/working/annotation/claims_master.csv
  /kaggle/working/annotation/to_annotate_annotator_A.csv
  /kaggle/working/annotation/to_annotate_annotator_B.csv
  /kaggle/working/annotation/to_annotate_annotator_C.csv


# SciRecQA -- Notebook 02 (agentic): Human Annotation & Inter-Annotator Agreement

**Benchmark**: SciRecQA v1.1
**Purpose**: Validate the auto-generated hallucination labels on a stratified
500-instance sample with 3 human annotators, compute inter-annotator agreement
(Cohen's kappa + Krippendorff's alpha), and write the verified labels back into
the benchmark. This is the quality anchor reviewers expect (paper Table 5).

This notebook is independent of the construction notebook (`agentic-nb1-...`).
It only reads the saved `scireqa_full_v1.1.jsonl`.

| Stage | Cell |
|---|---|
| Load benchmark + locate v1.1 files | 1 |
| Annotation rubric | 2 (markdown) |
| Stratified 500 sample (domain x difficulty) | 3 |
| Export per-annotator CSVs | 4 |
| (after annotation) Load completed CSVs + IAA | 6 |
| Write verified labels back | 7 |
| Table 5 summary for the paper | 8 |


In [7]:
# == 1. Imports + locate v1.1 benchmark files =================================
import json, os, random, itertools
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

random.seed(42); np.random.seed(42)

# locate the benchmark (prefer the RELATIONAL build; fall back to atomic)
def _find(name):
    for root in ["/kaggle/input", "..", "."]:
        r = Path(root)
        if r.exists():
            hits = list(r.rglob(name))
            if hits:
                return hits[0]
    return None
FULL_PATH = _find("scireqa_full_relational_v1.1.jsonl") or _find("scireqa_full_v1.1.jsonl")
assert FULL_PATH is not None, (
    "scireqa_full_relational_v1.1.jsonl (or _v1.1) not found -- attach the dataset / run agentic-nb1 first.")

# ANNOT_DIR: where existing inputs (sample_ids.json, claims_master.csv, done_annotator_*.csv)
# are read from -- may be a read-only Kaggle dataset attachment.
ANNOT_DIR = Path("/kaggle/input/datasets/joytuntonny/human-eval"); ANNOT_DIR.mkdir(exist_ok=True)
# OUT_DIR: where everything this notebook generates gets WRITTEN. Kaggle mounts
# /kaggle/input read-only, so if ANNOT_DIR points there, writes must go elsewhere.
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else ANNOT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)
N_SAMPLE     = 500
ANNOTATORS   = ["annotator_A", "annotator_B", "annotator_C"]
LABEL_SPACE  = ["supported", "unsupported", "neutral", "general_statement"]
TYPE_SPACE   = ["none", "entity", "factual", "relevance", "reasoning", "relational"]  # hallucination type (none = grounded)

from tqdm.auto import tqdm
# Belt-and-suspenders: an INDEPENDENT LLM judge (must differ from the Llama-3.3
# generator, to avoid self-preference bias) cross-checks the human-verified subset.
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
JUDGE_MODEL        = "qwen/qwen-2.5-72b-instruct:free"   # non-Llama judge ($0 free tier)
RUN_LLM_JUDGE      = False    # set True (with key) to run the 3-way agreement cross-check

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

benchmark = load_jsonl(FULL_PATH)
print(f"Loaded {len(benchmark)} instances from {FULL_PATH}")
print("Domains:", Counter(i["domain"] for i in benchmark))
print("With auto labels:", sum(1 for i in benchmark if i.get("hallucination_labels")))
print(f"ANNOT_DIR (read)  = {ANNOT_DIR.resolve()}")
print(f"OUT_DIR   (write) = {OUT_DIR.resolve()}")


Loaded 10000 instances from /kaggle/input/datasets/joytuntonny/rationale-subset/scireqa_full_relational_v1.1.jsonl
Domains: Counter({'arxiv_cs': 6000, 'acl_anthology': 2000, 'pubmed': 2000})
With auto labels: 10000
ANNOT_DIR (read)  = /kaggle/input/datasets/joytuntonny/human-eval
OUT_DIR   (write) = /kaggle/working


## 2. Annotation Rubric

Each row in the exported CSV is **one atomic claim** taken from a reference
rationale, shown together with the evidence (the ground-truth papers' titles +
abstract snippets). For every claim, assign exactly one label:

| Label | Definition |
|---|---|
| **supported** | The claim is directly entailed by the evidence (titles/abstracts). |
| **unsupported** | The claim contradicts the evidence, or asserts a specific fact (year, metric, author, title) that the evidence does **not** support (hallucination). |
| **neutral** | The claim is plausibly related but the evidence neither confirms nor denies it. |
| **general_statement** | Generic, non-verifiable prose (e.g. "these papers advance the field") with no checkable fact. |

Guidelines:
- Judge **only** against the shown evidence, not your own knowledge.
- A claim naming a number/year/metric/title that is absent from the evidence is **unsupported**, not neutral.
- Do not leave any row blank. Annotate independently (do not confer).

**Two columns to fill per row:**
1. `annotator_label` — the verdict: `supported` / `unsupported` / `neutral` / `general_statement`.
2. `annotator_type` — if the claim is a hallucination (`unsupported`), *which type*; otherwise `none`:

| Type | Definition |
|---|---|
| **entity** | A non-existent or wrong paper/author/title. |
| **factual** | A wrong atomic fact (year, metric, venue, result). |
| **relevance** | A recommended paper that does not match the user's interest. |
| **reasoning** | A fabricated *reason* for the recommendation. |
| **relational** | A wrong relationship *between papers* ("A builds on B", "A and B both address X", "A outperforms C") — check against the `relation` hint column; these often cannot be confirmed from any single abstract. |
| **none** | Grounded / not a hallucination. |

> The `relation` column shows an auto-detected relation type when the claim looks relational —
> verify whether that relationship actually holds, and mark `annotator_type = relational` if it does not.


**Relational claims (the focus of this study).** A relational claim asserts a relationship *between two papers* (e.g. "A builds on B", "A outperforms B", "A uses B's method"). Judge whether that relationship is **true**, using the abstracts and your domain knowledge — and consult the actual papers when abstracts are insufficient (inter-paper relations are usually not stated in a single abstract). Set `annotator_type = relational` and `annotator_label = unsupported` when the asserted relationship does **not** hold (a relational hallucination). **Do not** use any automated citation-graph tool to decide — your judgment is the independent gold against which the graph verifier (CiteCheck) is evaluated.


In [8]:
# == 3. Stratified 500-instance sample (domain x difficulty band) =============
# IDEMPOTENT: if a sample was already drawn and frozen (sample_ids.json exists --
# e.g. from the original run whose sample the done_annotator_*.csv files were built
# against), reuse it exactly instead of re-sampling. Re-sampling here is only safe
# if it reproduces bit-for-bit identically, which is not guaranteed across
# environments/library versions -- and a different sample would silently orphan
# any already-collected annotations.
_sample_ids_path = ANNOT_DIR / "sample_ids.json"
_by_id = {i["id"]: i for i in benchmark}

if _sample_ids_path.exists():
    sample_ids = json.load(open(_sample_ids_path, encoding="utf-8"))
    sample = [_by_id[i] for i in sample_ids if i in _by_id]
    missing = [i for i in sample_ids if i not in _by_id]
    print(f"Reusing frozen sample -> {_sample_ids_path} ({len(sample)} of {len(sample_ids)} ids found in benchmark)")
    if missing:
        print(f"  WARNING: {len(missing)} previously-sampled ids not found in the current "
              f"benchmark file (e.g. {missing[:3]}) -- benchmark file may have changed since the sample was frozen.")
else:
    # Only sample instances that actually carry auto labels (so there is something to verify).
    labeled = [i for i in benchmark if i.get("hallucination_labels")]
    print(f"Instances eligible (have labels): {len(labeled)}")

    # Difficulty bands via PER-DOMAIN terciles (robust to compressed score ranges;
    # absolute cutoffs would collapse every instance into one band).
    _dom_scores = defaultdict(list)
    for inst in labeled:
        s = inst.get("difficulty_score")
        if s is not None:
            _dom_scores[inst["domain"]].append(s)
    _dom_thresh = {}
    for d, vals in _dom_scores.items():
        if len(vals) >= 3:
            _dom_thresh[d] = tuple(np.quantile(vals, [1/3, 2/3]))
        else:
            _dom_thresh[d] = (0.33, 0.66)

    def difficulty_band(inst):
        s = inst.get("difficulty_score")
        if s is None:
            return "unknown"
        q1, q2 = _dom_thresh.get(inst["domain"], (0.33, 0.66))
        if s <= q1:  return "easy"
        if s <= q2:  return "medium"
        return "hard"

    # Bucket by (domain, band)
    buckets = defaultdict(list)
    for inst in labeled:
        buckets[(inst["domain"], difficulty_band(inst))].append(inst)

    domains = sorted({d for (d, _) in buckets})
    per_domain = max(1, N_SAMPLE // max(1, len(domains)))

    sample = []
    for dom in domains:
        dom_bucket_keys = [k for k in buckets if k[0] == dom]
        if not dom_bucket_keys:
            continue
        per_band = max(1, per_domain // len(dom_bucket_keys))
        for k in dom_bucket_keys:
            pool = buckets[k][:]
            random.shuffle(pool)
            sample.extend(pool[:per_band])

    # Trim / top-up to exactly N_SAMPLE
    random.shuffle(sample)
    sample = sample[:N_SAMPLE]
    if len(sample) < N_SAMPLE:
        seen = {i["id"] for i in sample}
        extra = [i for i in labeled if i["id"] not in seen]
        random.shuffle(extra)
        sample.extend(extra[:N_SAMPLE - len(sample)])

    sample_ids = [i["id"] for i in sample]
    _sample_ids_out = OUT_DIR / "sample_ids.json"
    with open(_sample_ids_out, "w", encoding="utf-8") as f:
        json.dump(sample_ids, f, indent=2)
    print(f"Sampled {len(sample)} instances (fresh) -> saved {_sample_ids_out}")

print(f"Sample size: {len(sample)}")
print("  by domain :", Counter(i["domain"] for i in sample))


Reusing frozen sample -> /kaggle/input/datasets/joytuntonny/human-eval/sample_ids.json (500 of 500 ids found in benchmark)
Sample size: 500
  by domain : Counter({'acl_anthology': 168, 'arxiv_cs': 167, 'pubmed': 165})


In [9]:
# == 4. Export per-annotator CSVs (one row per atomic claim) ==================
# IDEMPOTENT: if claims_master.csv already exists (frozen from the original run),
# reload it instead of rebuilding -- this is the exact master key the done_annotator_*.csv
# files were annotated against, so it must not drift.
_claims_master_path = ANNOT_DIR / "claims_master.csv"

if _claims_master_path.exists():
    claims_df = pd.read_csv(_claims_master_path, encoding="utf-8")
    print(f"Reusing frozen master key -> {_claims_master_path} ({len(claims_df)} claim rows, "
          f"{claims_df['instance_id'].nunique()} instances)")
else:
    def evidence_snippet(inst, max_papers=5, per=240):
        parts = []
        for p in inst["ground_truth_papers"][:max_papers]:
            t = (p.get("title", "") or "").strip()
            a = (p.get("abstract", "") or "")[:per].strip()
            parts.append(f"- {t}: {a}")
        return "\n".join(parts)

    rows = []
    for inst in sample:
        ev = evidence_snippet(inst)
        for lbl in inst["hallucination_labels"]:
            rows.append({
                "instance_id":  inst["id"],
                "domain":       inst["domain"],
                "claim_id":     lbl["claim_id"],
                "claim_text":   lbl["claim_text"],
                "claim_type":   lbl.get("claim_type", ""),
                "relation":     (lbl.get("relational") or {}).get("relation", ""),  # auto hint; verify if relational
                "auto_verdict": lbl["verdict"],          # for reference / adjudication only
                "evidence":     ev,
                "annotator_label": "",                   # <-- verdict: supported/unsupported/neutral/general_statement
                "annotator_type":  "",                   # <-- if unsupported: entity/factual/relevance/reasoning/relational ; else none
            })

    claims_df = pd.DataFrame(rows)
    print(f"Total claim rows to annotate: {len(claims_df)}  "
          f"(over {claims_df['instance_id'].nunique()} instances)")

    # Blank template (auto_verdict hidden to avoid anchoring bias) for each annotator
    template = claims_df.drop(columns=["auto_verdict"]).copy()
    for ann in ANNOTATORS:
        out = OUT_DIR / f"to_annotate_{ann}.csv"
        template.to_csv(out, index=False, encoding="utf-8")
        print(f"  exported {out}")

    _claims_master_out = OUT_DIR / "claims_master.csv"
    claims_df.to_csv(_claims_master_out, index=False, encoding="utf-8")
    print(f"  exported {_claims_master_out} (master key with auto_verdict)")
    print("\nHand each to_annotate_*.csv to a different annotator; they fill the "
          "'annotator_label' column with one of:", LABEL_SPACE)


Reusing frozen master key -> /kaggle/input/datasets/joytuntonny/human-eval/claims_master.csv (2216 claim rows, 500 instances)


## 5. >>> Pause: collect human annotations <<<

Give each `annotation/to_annotate_<annotator>.csv` to a different person. Each fills the
`annotator_label` column for every row using the rubric in Section 2, then saves the
file back as `annotation/done_<annotator>.csv` (same rows, same order).

When all three `done_*.csv` files are present, continue with Section 6.

A tiny synthetic fallback is provided in Section 6 so the IAA code can be smoke-tested
before real annotations exist (it copies the auto labels with light noise). Delete /
ignore that block once real annotations are in.


In [10]:
# == 6. Load completed annotations + Inter-Annotator Agreement ===============
# NOTE: supports PARTIAL annotation checkpoints (fewer than N_SAMPLE instances done).
# "common" below is the set of claims where all 3 annotators actually filled in a
# label -- not just every row that exists in the CSV template.
from sklearn.metrics import cohen_kappa_score

def _norm_claim_id(x):
    """Normalize claim_id for joining: '000' and '0' and 0 must all compare equal,
    since the benchmark file's claim_id formatting (zero-padded) can drift from
    claims_master.csv's (unpadded) after regeneration/relabeling runs."""
    try:
        return str(int(x))
    except (TypeError, ValueError):
        return str(x).strip()

def _find_done(ann):
    """Locate done_<ann>.csv. Tries the canonical ANNOT_DIR path first, then falls
    back to an rglob search so this still works if the annotation folder was
    uploaded under a different name/nesting (e.g. a Kaggle dataset attachment)."""
    fname = f"done_{ann}.csv"
    direct = ANNOT_DIR / fname
    if direct.exists():
        return direct
    for root in [ANNOT_DIR, Path(".."), Path("."), Path("/kaggle/input")]:
        if root.exists():
            hits = list(root.rglob(fname))
            if hits:
                return hits[0]
    return direct  # missing; caller's .exists() check below will catch it

done_paths = {ann: _find_done(ann) for ann in ANNOTATORS}
have_real  = all(p.exists() for p in done_paths.values())

if have_real:
    ann_frames = {ann: pd.read_csv(p, encoding="utf-8") for ann, p in done_paths.items()}
    print("Loaded real annotations from:")
    for a, p in done_paths.items():
        print(f"  {a}: {p}  ({len(ann_frames[a])} rows)")
else:
    print("WARNING -- could not find all three done_annotator_*.csv files:")
    for a, p in done_paths.items():
        print(f"  {a}: {p}  (exists={p.exists()})")
    # ---- SMOKE-TEST FALLBACK (synthetic) -- remove once real annotations exist ----
    print("-> generating SYNTHETIC annotations for a smoke test only.")
    rng = np.random.default_rng(0)
    ann_frames = {}
    base = claims_df.copy()
    for s, ann in enumerate(ANNOTATORS):
        d = base.drop(columns=["auto_verdict", "evidence"]).copy()
        verds, types = [], []
        for _, row in base.iterrows():
            v = row["auto_verdict"]; v = v if v in LABEL_SPACE else "neutral"
            if rng.random() < 0.15:
                v = rng.choice(LABEL_SPACE)
            verds.append(v)
            if str(row.get("relation", "")):
                types.append("relational")
            elif v == "unsupported":
                types.append(rng.choice(["entity", "factual", "relevance", "reasoning"]))
            else:
                types.append("none")
        d["annotator_label"] = verds
        d["annotator_type"]  = types
        ann_frames[ann] = d

# Align on (instance_id, claim_id)
def key_series(df):
    return df.assign(_k=df["instance_id"].astype(str) + "|" + df["claim_id"].apply(_norm_claim_id)) \
             .set_index("_k")["annotator_label"]

def _nonblank_index(s):
    s2 = s.dropna()                                   # unfilled CSV cells parse as NaN (float), not ""
    s2 = s2[s2.astype(str).str.strip() != ""]          # also catch any literal empty-string cells
    return set(s2.index)

series = {ann: key_series(df) for ann, df in ann_frames.items()}
# Only claims where ALL annotators filled in a real (non-blank) label -- this is what
# makes IAA correct on a partial checkpoint, since every row exists in the CSV
# template regardless of whether it has been annotated yet.
filled_idx = {ann: _nonblank_index(s) for ann, s in series.items()}
common = sorted(set.intersection(*filled_idx.values()))
n_total_rows = len(next(iter(series.values())))
print(f"\nClaims with all {len(ANNOTATORS)} annotations filled in: {len(common)} of {n_total_rows} total rows")
n_inst_done = len({k.split('|')[0] for k in common})
print(f"Instances represented in that set: {n_inst_done} of {N_SAMPLE} sampled "
      f"({N_SAMPLE - n_inst_done} instances still unannotated)")
aligned = {ann: series[ann].reindex(common).tolist() for ann in ANNOTATORS}

# ---- Pairwise Cohen's kappa ----
print("\nPairwise Cohen's kappa:")
kappas = []
for a, b in itertools.combinations(ANNOTATORS, 2):
    k = cohen_kappa_score(aligned[a], aligned[b], labels=LABEL_SPACE)
    kappas.append(k)
    print(f"  {a} vs {b}: kappa = {k:.3f}")
print(f"  mean pairwise kappa = {np.mean(kappas):.3f}")

# ---- Krippendorff's alpha (nominal), dependency-free ----
def krippendorff_alpha_nominal(data_by_coder):
    """data_by_coder: dict coder -> list of labels (None for missing). Nominal metric."""
    coders = list(data_by_coder)
    n_units = len(next(iter(data_by_coder.values())))
    # units x coders matrix
    units = []
    for u in range(n_units):
        vals = [data_by_coder[c][u] for c in coders if data_by_coder[c][u] is not None]
        if len(vals) >= 2:
            units.append(vals)
    if not units:
        return float("nan")
    # observed disagreement
    classes = sorted({v for u in units for v in u})
    Do_num = 0.0; Do_den = 0.0
    for u in units:
        m = len(u)
        cnt = Counter(u)
        pair_total = m * (m - 1)
        disagree = pair_total - sum(c * (c - 1) for c in cnt.values())
        Do_num += disagree
        Do_den += pair_total
    Do = Do_num / Do_den
    # expected disagreement
    total = Counter(v for u in units for v in u)
    N = sum(total.values())
    De = 1.0 - sum(c * (c - 1) for c in total.values()) / (N * (N - 1))
    return 1.0 - (Do / De) if De > 0 else float("nan")

alpha = krippendorff_alpha_nominal({ann: aligned[ann] for ann in ANNOTATORS})
print(f"\nKrippendorff's alpha (nominal): {alpha:.3f}")

iaa_summary = {"mean_cohen_kappa": round(float(np.mean(kappas)), 3),
               "pairwise_kappa": {f"{a}|{b}": round(float(cohen_kappa_score(aligned[a], aligned[b], labels=LABEL_SPACE)), 3)
                                   for a, b in itertools.combinations(ANNOTATORS, 2)},
               "krippendorff_alpha": round(float(alpha), 3),
               "n_claims": len(common), "n_instances": n_inst_done,
               "n_instances_sampled": N_SAMPLE, "synthetic": not have_real}
with open(OUT_DIR / "iaa_summary.json", "w", encoding="utf-8") as f:
    json.dump(iaa_summary, f, indent=2)
print(f"Saved {OUT_DIR / 'iaa_summary.json'}")

# ---- Per-TYPE IAA (the resource novelty: typed hallucination annotation) ----
if all("annotator_type" in df.columns for df in ann_frames.values()):
    def _ts(df):
        return df.assign(_k=df["instance_id"].astype(str) + "|" + df["claim_id"].apply(_norm_claim_id)) \
                 .set_index("_k")["annotator_type"]
    tser     = {ann: _ts(df) for ann, df in ann_frames.items()}
    tfilled  = {ann: _nonblank_index(s) for ann, s in tser.items()}
    tcommon  = sorted(set.intersection(*tfilled.values()) & set(common))
    talign   = {ann: tser[ann].reindex(tcommon).tolist() for ann in ANNOTATORS}
    tk = [cohen_kappa_score(talign[a], talign[b], labels=TYPE_SPACE)
          for a, b in itertools.combinations(ANNOTATORS, 2)]
    talpha = krippendorff_alpha_nominal({ann: talign[ann] for ann in ANNOTATORS})
    # relational-vs-rest (binary) agreement -- the focus type
    binr = {ann: ["relational" if x == "relational" else "other" for x in talign[ann]] for ann in ANNOTATORS}
    bk = [cohen_kappa_score(binr[a], binr[b], labels=["relational", "other"])
          for a, b in itertools.combinations(ANNOTATORS, 2)]
    print(f"\nPer-type IAA      : mean kappa={np.mean(tk):.3f}, alpha={talpha:.3f}")
    print(f"Relational (binary): mean kappa={np.mean(bk):.3f}")
    iaa_summary["type"] = {"mean_cohen_kappa": round(float(np.mean(tk)), 3),
                           "krippendorff_alpha": round(float(talpha), 3),
                           "relational_binary_kappa": round(float(np.mean(bk)), 3),
                           "n_claims": len(tcommon)}
    with open(OUT_DIR / "iaa_summary.json", "w", encoding="utf-8") as f:
        json.dump(iaa_summary, f, indent=2)
    print(f"Updated {OUT_DIR / 'iaa_summary.json'} with per-type agreement")
else:
    print("\n(no annotator_type column -> per-type IAA skipped)")


Loaded real annotations from:
  annotator_A: /kaggle/input/datasets/joytuntonny/human-eval/done_annotator_A.csv  (2216 rows)
  annotator_B: /kaggle/input/datasets/joytuntonny/human-eval/done_annotator_B.csv  (2216 rows)
  annotator_C: /kaggle/input/datasets/joytuntonny/human-eval/done_annotator_C.csv  (2216 rows)

Claims with all 3 annotations filled in: 303 of 2216 total rows
Instances represented in that set: 68 of 500 sampled (432 instances still unannotated)

Pairwise Cohen's kappa:
  annotator_A vs annotator_B: kappa = 0.841
  annotator_A vs annotator_C: kappa = 0.734
  annotator_B vs annotator_C: kappa = 0.818
  mean pairwise kappa = 0.798

Krippendorff's alpha (nominal): 0.797
Saved /kaggle/working/iaa_summary.json

Per-type IAA      : mean kappa=0.553, alpha=0.547
Relational (binary): mean kappa=0.575
Updated /kaggle/working/iaa_summary.json with per-type agreement


In [11]:
# == 6.5 DIAGNOSTIC -- why did cell 7 verify 0 claims? =========================
# Run this if cell 7 prints "Human-verified: 0 claims across 0 instances".
# It checks, step by step, where the (instance_id, claim_id) keys stop matching
# between: sample_ids.json -> benchmark instances -> benchmark claim_ids -> common.

bench_ids = {inst["id"] for inst in benchmark}
sample_id_set_dbg = set(sample_ids)
print(f"sample_ids.json entries        : {len(sample_id_set_dbg)}")
print(f"benchmark instances (this run) : {len(bench_ids)}")
overlap_inst = sample_id_set_dbg & bench_ids
print(f"sample_ids that exist in THIS benchmark file : {len(overlap_inst)} of {len(sample_id_set_dbg)}")
if overlap_inst:
    print("  example matched id:", next(iter(overlap_inst)))
else:
    print("  !! ZERO instance-id overlap -- FULL_PATH resolved to a DIFFERENT benchmark file")
    print("     than the one sample_ids.json/claims_master.csv were built from.")
    print("  sample_ids.json example ids:", list(sample_id_set_dbg)[:3])
    print("  benchmark example ids      :", list(bench_ids)[:3])
    print("  FULL_PATH used this run    :", FULL_PATH)

# Now check claim-level keys, restricted to instances that DID match above.
bench_keys = set()
for inst in benchmark:
    if inst["id"] not in overlap_inst:
        continue
    for lbl in inst.get("hallucination_labels", []):
        bench_keys.add(f"{inst['id']}|{lbl['claim_id']}")
print(f"\nbenchmark claim keys (id|claim_id) for matched instances: {len(bench_keys)}")
print(f"common (from cell 6, annotated + filled)                : {len(common)}")
overlap_claims = bench_keys & set(common)
print(f"overlap between the two                                  : {len(overlap_claims)}")
if common and not overlap_claims:
    print("  !! instance ids matched but claim_ids did NOT -- the benchmark's")
    print("     hallucination_labels/claim_id values differ from claims_master.csv's.")
    print("  claims_master.csv example key:", common[0])
    ex_inst = next(iter(overlap_inst)) if overlap_inst else None
    if ex_inst:
        ex_claim_ids = [l["claim_id"] for i in benchmark if i["id"] == ex_inst for l in i.get("hallucination_labels", [])]
        print(f"  benchmark claim_ids for that same instance ({ex_inst}):", ex_claim_ids[:10])


sample_ids.json entries        : 500
benchmark instances (this run) : 10000
sample_ids that exist in THIS benchmark file : 500 of 500
  example matched id: dbdbff4a913d

benchmark claim keys (id|claim_id) for matched instances: 2216
common (from cell 6, annotated + filled)                : 303
overlap between the two                                  : 0
  !! instance ids matched but claim_ids did NOT -- the benchmark's
     hallucination_labels/claim_id values differ from claims_master.csv's.
  claims_master.csv example key: 001035c4fa36|0
  benchmark claim_ids for that same instance (dbdbff4a913d): ['000', '001', '002', '003', '004']


In [12]:
# == 7. Majority-vote adjudication + write verified labels back ===============
# Majority label per claim across the 3 annotators; ties -> fall back to auto_verdict.
master = claims_df.set_index(
    claims_df["instance_id"].astype(str) + "|" + claims_df["claim_id"].apply(_norm_claim_id))

def majority(labels):
    c = Counter(labels)
    top, n = c.most_common(1)[0]
    if list(c.values()).count(n) > 1:      # tie
        return None
    return top

human_label_for = {}
for k in common:
    votes = [series[ann].get(k) for ann in ANNOTATORS]
    votes = [v for v in votes if isinstance(v, str)]
    mj = majority(votes)
    if mj is None and k in master.index:
        mj = master.loc[k, "auto_verdict"]
    human_label_for[k] = mj

# Majority hallucination-TYPE per claim (per-type gold + the relational finding)
human_type_for = {}
if all("annotator_type" in df.columns for df in ann_frames.values()):
    _tser = {ann: df.assign(_k=df["instance_id"].astype(str) + "|" + df["claim_id"].apply(_norm_claim_id))
                    .set_index("_k")["annotator_type"] for ann, df in ann_frames.items()}
    for k in common:
        votes = [_tser[ann].get(k) for ann in ANNOTATORS]
        votes = [v for v in votes if isinstance(v, str)]
        mj = majority(votes)
        human_type_for[k] = mj if mj is not None else "none"

# Write back into the benchmark instances (only the sampled subset)
# NOTE: keys are normalized the same way as cell 6 (_norm_claim_id) since the
# benchmark file's claim_id formatting can differ (e.g. zero-padded "000" vs
# claims_master.csv's unpadded "0") from a later regeneration/relabeling run.
sample_id_set = set(sample_ids)
n_claims_verified = 0; n_inst_verified = 0
for inst in benchmark:
    if inst["id"] not in sample_id_set:
        continue
    touched = False
    for lbl in inst.get("hallucination_labels", []):
        k = f"{inst['id']}|{_norm_claim_id(lbl['claim_id'])}"
        if k in human_label_for and human_label_for[k] is not None:
            lbl["human_verified"] = True
            lbl["human_label"]    = human_label_for[k]
            lbl["human_type"]     = human_type_for.get(k)
            n_claims_verified += 1
            touched = True
    if touched:
        inst["annotation_status"] = "human_verified"
        n_inst_verified += 1

print(f"Human-verified: {n_claims_verified} claims across {n_inst_verified} instances")

# Save: (a) the verified subset, (b) the full benchmark with verifications merged in
verified_subset = [i for i in benchmark if i["id"] in sample_id_set and i["annotation_status"] == "human_verified"]
def save_jsonl(items, path):
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")
    print(f"Saved {len(items)} -> {path}")

save_jsonl(verified_subset, OUT_DIR / "scireqa_human_verified_subset.jsonl")
save_jsonl(benchmark, OUT_DIR / (FULL_PATH.stem + "_humanmerged.jsonl"))


Human-verified: 303 claims across 68 instances
Saved 68 -> /kaggle/working/scireqa_human_verified_subset.jsonl
Saved 10000 -> /kaggle/working/scireqa_full_relational_v1.1_humanmerged.jsonl


In [13]:
# == 7.5 Calibrate MiniCheck thresholds on human gold + (optional) LLM judge ====
# (a) Threshold calibration: MiniCheck stores its RAW support prob in 'confidence';
#     sweep (support, unsupport) thresholds to best match the human gold labels.
cal = []   # (support_prob, human_label)
for inst in verified_subset:
    for l in inst["hallucination_labels"]:
        if (l.get("human_verified") and l.get("human_label")
                and str(l.get("label_type", "")).startswith("auto_minicheck")):
            cal.append((float(l["confidence"]), l["human_label"]))

if cal:
    def _verdict(pr, sup, uns):
        return "supported" if pr >= sup else ("unsupported" if pr < uns else "neutral")
    best = None
    for sup in [round(float(x), 2) for x in np.arange(0.40, 0.75, 0.05)]:
        for uns in [round(float(x), 2) for x in np.arange(0.10, 0.40, 0.05)]:
            if uns >= sup:
                continue
            preds = [_verdict(pr, sup, uns) for pr, _ in cal]
            golds = [g for _, g in cal]
            acc   = float(np.mean([a == b for a, b in zip(preds, golds)]))
            try:
                k = cohen_kappa_score(preds, golds, labels=LABEL_SPACE)
            except Exception:
                k = float("nan")
            if best is None or (k == k and k > best[0]):   # k==k skips NaN
                best = (k, acc, sup, uns)
    print(f"Calibration on {len(cal)} verified factual claims:")
    print(f"  best thresholds -> SUPPORT={best[2]}  UNSUPPORT={best[3]}  "
          f"(kappa={best[0]:.3f}, acc={best[1]:.3f})")
    print(f"  ACTION: set MINICHECK_SUPPORT_THRESHOLD={best[2]} and "
          f"MINICHECK_UNSUPPORT_THRESHOLD={best[3]} in agentic-nb1, then re-run labeling.")
    with open(OUT_DIR / "minicheck_calibration.json", "w", encoding="utf-8") as f:
        json.dump({"support_threshold": best[2], "unsupport_threshold": best[3],
                   "kappa_vs_human": round(float(best[0]), 3), "n_claims": len(cal)}, f, indent=2)
else:
    print("No MiniCheck-labelled verified claims found "
          "(run agentic-nb1 with LABELER='minicheck' first).")

# (b) Optional independent LLM-judge -> 3-way agreement (MiniCheck <-> judge <-> human)
def _evidence_for(inst, per=240, k=5):
    return "\n".join(f"- {(p.get('title','') or '').strip()}: {(p.get('abstract','') or '')[:per].strip()}"
                     for p in inst["ground_truth_papers"][:k])

if RUN_LLM_JUDGE and OPENROUTER_API_KEY:
    import openai
    _client = openai.OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")

    def _judge(claim, evidence):
        prompt = (f"Evidence:\n{evidence}\n\nClaim: {claim}\n\n"
                  "Is the claim supported by the evidence? "
                  "Answer with exactly one word: supported, unsupported, or neutral.")
        try:
            r = _client.chat.completions.create(
                model=JUDGE_MODEL, temperature=0.0, max_tokens=4,
                messages=[{"role": "user", "content": prompt}])
            ans = (r.choices[0].message.content or "").strip().lower()
            for v in ["unsupported", "supported", "neutral"]:
                if v in ans:
                    return v
        except Exception as e:
            print("judge error:", e)
        return "neutral"

    mc, ju, hu = [], [], []
    for inst in tqdm(verified_subset, desc="LLM-judge cross-check"):
        ev = _evidence_for(inst)
        for l in inst["hallucination_labels"]:
            if (l.get("human_verified") and l.get("human_label")
                    and l.get("claim_type") == "factual_candidate"):
                mc.append(l["verdict"])                 # MiniCheck verdict
                hu.append(l["human_label"])             # human gold
                ju.append(_judge(l["claim_text"], ev))  # independent judge
    if mc:
        def _k(a, b): return float(cohen_kappa_score(a, b, labels=LABEL_SPACE))
        res = {"minicheck_vs_human": round(_k(mc, hu), 3),
               "judge_vs_human":     round(_k(ju, hu), 3),
               "minicheck_vs_judge": round(_k(mc, ju), 3),
               "n_claims": len(mc), "judge_model": JUDGE_MODEL}
        print(f"\n3-way agreement on {len(mc)} verified factual claims:")
        print(f"  MiniCheck vs Human : kappa = {res['minicheck_vs_human']}")
        print(f"  Judge     vs Human : kappa = {res['judge_vs_human']}")
        print(f"  MiniCheck vs Judge : kappa = {res['minicheck_vs_judge']}")
        with open(OUT_DIR / "threeway_agreement.json", "w", encoding="utf-8") as f:
            json.dump(res, f, indent=2)
        print(f"Saved {OUT_DIR / 'threeway_agreement.json'}")
else:
    print("\nLLM-judge cross-check skipped (set RUN_LLM_JUDGE=True and OPENROUTER_API_KEY).")


Calibration on 251 verified factual claims:
  best thresholds -> SUPPORT=0.5  UNSUPPORT=0.1  (kappa=0.022, acc=0.279)
  ACTION: set MINICHECK_SUPPORT_THRESHOLD=0.5 and MINICHECK_UNSUPPORT_THRESHOLD=0.1 in agentic-nb1, then re-run labeling.

LLM-judge cross-check skipped (set RUN_LLM_JUDGE=True and OPENROUTER_API_KEY).


In [14]:
# == 8. Table 5 -- human validation summary (for the paper) ==================
ver_lbls = [l for i in verified_subset for l in i["hallucination_labels"] if l.get("human_verified")]
auto_vs_human_agree = sum(1 for l in ver_lbls if l["verdict"] == l["human_label"])

rows = []
for dom in sorted({i["domain"] for i in verified_subset}):
    di = [i for i in verified_subset if i["domain"] == dom]
    dl = [l for i in di for l in i["hallucination_labels"] if l.get("human_verified")]
    agree = sum(1 for l in dl if l["verdict"] == l["human_label"])
    rows.append({
        "domain": dom,
        "instances": len(di),
        "verified_claims": len(dl),
        "auto==human (%)": round(100 * agree / max(1, len(dl)), 1),
        "human_unsupported (%)": round(100 * sum(1 for l in dl if l["human_label"] == "unsupported") / max(1, len(dl)), 1),
    })
table5 = pd.DataFrame(rows)

print("Table 5 -- SciRecQA v1.1 human validation")
print(table5.to_string(index=False))
print(f"\nOverall auto-vs-human agreement: "
      f"{100*auto_vs_human_agree/max(1,len(ver_lbls)):.1f}%  over {len(ver_lbls)} claims")
print("IAA:", json.load(open(OUT_DIR / 'iaa_summary.json')))

table5.to_csv(OUT_DIR / "table5_human_validation.csv", index=False, encoding="utf-8")
print(f"Saved {OUT_DIR / 'table5_human_validation.csv'}")


Table 5 -- SciRecQA v1.1 human validation
       domain  instances  verified_claims  auto==human (%)  human_unsupported (%)
acl_anthology         26              113             26.5                   31.0
     arxiv_cs         19               88             25.0                   28.4
       pubmed         23              102             31.4                   20.6

Overall auto-vs-human agreement: 27.7%  over 303 claims
IAA: {'mean_cohen_kappa': 0.798, 'pairwise_kappa': {'annotator_A|annotator_B': 0.841, 'annotator_A|annotator_C': 0.734, 'annotator_B|annotator_C': 0.818}, 'krippendorff_alpha': 0.797, 'n_claims': 303, 'n_instances': 68, 'n_instances_sampled': 500, 'synthetic': False, 'type': {'mean_cohen_kappa': 0.553, 'krippendorff_alpha': 0.547, 'relational_binary_kappa': 0.575, 'n_claims': 303}}
Saved /kaggle/working/table5_human_validation.csv
